<a href="https://colab.research.google.com/github/camgenomicmedicine/GMO4/blob/main/GMO4_Practical_8_Pipelines_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GMO4 Advanced Bioinformatics - Practical 8
## Building reproducible NGS pipelines in Google Colab

In Practicals 2-5 you ran the NGS analysis one command at a time: FastQC, Trimmomatic, BWA, samtools, Picard, GATK, FreeBayes and VCFtools. In this practical you will run the same analysis as a pipeline.

This version of the notebook is designed for **Google Colab**, so it installs the required command-line tools into the temporary Colab runtime. Colab sessions are reset periodically, so rerun the installation and setup cells whenever the runtime restarts.

The course data archive is not embedded in this notebook. Upload the GMO4 course data `.tar`, `.tar.gz` or `.tgz` archive into `/content` using the Colab file manager before running the data setup cell.

Work through the cells in order. The accompanying practical book contains the explanations and the questions to answer in Canvas.


## 0. Install command-line tools

This cell installs the command-line tools required for the pipeline practical into a Colab environment at `/content/bioinfo_env`.

The installed tools include:

- `fastqc`
- `trimmomatic`
- `bwa`
- `samtools`
- `bcftools`
- `picard`
- `gatk`
- `freebayes`
- `vcftools`
- `qualimap`
- `snakemake`
- `graphviz` / `dot`

Run this cell once at the start of a Colab session. It may take several minutes.


In [ ]:
%%bash
set -euo pipefail

mkdir -p /content/bin

if [ ! -x /content/bin/micromamba ]; then
  echo "Downloading micromamba..."
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest \
    | tar -xvj -C /content/bin --strip-components=1 bin/micromamba
else
  echo "micromamba is already installed."
fi

export MAMBA_ROOT_PREFIX=/content/micromamba
ENV_PREFIX=/content/bioinfo_env

if [ ! -x "$ENV_PREFIX/bin/samtools" ]; then
  echo "Creating bioinformatics environment..."
  /content/bin/micromamba create -y -p "$ENV_PREFIX" \
    -c conda-forge -c bioconda \
    python=3.11 \
    fastqc trimmomatic bwa samtools bcftools picard gatk4 freebayes vcftools \
    qualimap snakemake-minimal graphviz r-base openjdk
else
  echo "Bioinformatics environment already exists."
fi

echo "Setup complete."


In [ ]:
# Put the Colab bioinformatics environment on PATH for Python subprocess calls.
import os
ENV_PREFIX = "/content/bioinfo_env"
os.environ["PATH"] = f"{ENV_PREFIX}/bin:/content/bin:" + os.environ["PATH"]
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"

print("PATH begins with:")
for entry in os.environ["PATH"].split(":")[:5]:
    print(" -", entry)


### Check installed tools

This cell checks that the expected programs are available on the Colab `PATH`.


In [ ]:
%%bash
set -euo pipefail
export PATH="/content/bioinfo_env/bin:/content/bin:$PATH"

echo "Checking installed tools..."
for tool in fastqc trimmomatic bwa samtools bcftools picard gatk freebayes vcftools qualimap snakemake dot; do
  printf "%-14s" "$tool"
  command -v "$tool"
done

echo
echo "Selected versions:"
fastqc --version || true
samtools --version | sed -n '1p'
bcftools --version | sed -n '1p'
gatk --version | sed -n '1,2p' || true
freebayes --version || true
vcftools --version || true
snakemake --version || true


## 1. Prepare the course data

Upload the GMO4 course data archive into `/content` using the Colab file manager before running the next cell.

The archive should contain:

- `lane1/s-7-1.fastq`
- `lane1/s-7-2.fastq`
- `lane2/s-7-1.fastq`
- `lane2/s-7-2.fastq`
- `Saccharomyces_cerevisiae.EF4.68.dna.toplevel.fa`
- `primers_adapters.fa`

The setup cell extracts the archive, finds the data directory, and copies the required files into a clean working directory called `/content/Practical8_pipeline_work`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, textwrap, tarfile

# Keep the conda environment visible to all subprocess calls from Python cells.
ENV_PREFIX = "/content/bioinfo_env"
os.environ["PATH"] = f"{ENV_PREFIX}/bin:/content/bin:" + os.environ["PATH"]

os.chdir("/content")

ref_name = "Saccharomyces_cerevisiae.EF4.68.dna.toplevel.fa"
required_relative_paths = [
    "lane1/s-7-1.fastq",
    "lane1/s-7-2.fastq",
    "lane2/s-7-1.fastq",
    "lane2/s-7-2.fastq",
    ref_name,
    "primers_adapters.fa",
]

candidate_names = [
    "GM04_AdvBioinfo_NGS_April2026.tar",
    "GM04_AdvBioinfo_NGS_Mar2026.tar",
    "GM04_AdvBioinfo_NGS.tar",
    "GM04_AdvBioinfo_NGS_April2021.tar",
]

def looks_like_data_dir(path):
    path = Path(path).expanduser().resolve()
    return all((path / rel).exists() for rel in required_relative_paths)

def safe_extract(tar, path):
    """Extract a tar archive while preventing path traversal outside the destination."""
    dest = path.resolve()
    for member in tar.getmembers():
        target = (path / member.name).resolve()
        try:
            target.relative_to(dest)
        except ValueError:
            raise RuntimeError(f"Unsafe path in archive: {member.name}")
    tar.extractall(path)

# Look for an archive uploaded into /content.
archive_path = None
for name in candidate_names:
    p = Path("/content") / name
    if p.exists():
        archive_path = p
        break

if archive_path is None:
    archives = sorted(
        list(Path("/content").glob("*.tar")) +
        list(Path("/content").glob("*.tar.gz")) +
        list(Path("/content").glob("*.tgz"))
    )
    if len(archives) == 1:
        archive_path = archives[0]
    elif len(archives) > 1:
        raise FileNotFoundError(
            "More than one archive was found in /content:\n"
            + "\n".join(f" - {p.name}" for p in archives)
            + "\nPlease keep only the GMO4 course data archive in /content, then rerun this cell."
        )

if archive_path is None or not archive_path.exists():
    raise FileNotFoundError(
        "No GMO4 course data archive was found in /content. "
        "Use the Colab file manager to upload the .tar, .tar.gz or .tgz archive, then rerun this cell."
    )

print("Using archive:", archive_path)

extract_dir = Path("/content/gmo4_data")
extract_dir.mkdir(exist_ok=True)

marker = extract_dir / ".extracted_from"
archive_marker = str(archive_path.resolve())

if not marker.exists() or marker.read_text().strip() != archive_marker:
    print("Extracting archive...")
    with tarfile.open(archive_path, "r:*") as tar:
        safe_extract(tar, extract_dir)
    marker.write_text(archive_marker)
else:
    print("Archive already extracted.")

# Find the directory containing the expected input files.
source_dir = None
for candidate in [extract_dir] + [p for p in extract_dir.rglob("*") if p.is_dir()]:
    if looks_like_data_dir(candidate):
        source_dir = candidate.resolve()
        break

if source_dir is None:
    raise FileNotFoundError(
        "Could not find the expected VariantCalling data directory inside the archive. "
        "Check that the correct GMO4 data archive was uploaded."
    )

workdir = Path("/content/Practical8_pipeline_work").resolve()
workdir.mkdir(exist_ok=True)

# Copy the required files into a clean working directory.
for rel in required_relative_paths:
    src = source_dir / rel
    dst = workdir / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() or src.stat().st_size != dst.stat().st_size:
        shutil.copy2(src, dst)

os.environ["WORKDIR"] = str(workdir)
os.chdir(workdir)

print("Source data directory:", source_dir)
print("Working directory:", Path.cwd())
print("Files in working directory:")
for item in sorted(Path.cwd().iterdir()):
    print(" -", item.name)


In [ ]:
# Helper function used throughout the notebook.
from pathlib import Path
import subprocess, sys, os, textwrap, shutil

ENV_PREFIX = "/content/bioinfo_env"
os.environ["PATH"] = f"{ENV_PREFIX}/bin:/content/bin:" + os.environ["PATH"]

def run(cmd, cwd=None, check=True):
    """Run a shell command from a notebook cell and show stdout/stderr."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        cwd=cwd or Path.cwd(),
        shell=True,
        executable="/bin/bash",
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode, cmd, output=result.stdout, stderr=result.stderr
        )
    return result

print("Helper function ready. Current working directory:", Path.cwd())


**Question 1.** Which directory is your working directory? Why is it safer to run the pipeline in a copy of the data rather than directly in the original `VariantCalling` directory?


## 2. Simple scripts as pipelines

A pipeline can be as simple as a script that runs one command after another. The first examples below are deliberately limited: they use one FASTQ file only, so they do not represent the full paired-end workflow. Their purpose is to help you recognise how a script becomes a pipeline and where scripted pipelines become difficult to maintain.


In [ ]:
scripts_dir = Path("script_examples")
scripts_dir.mkdir(exist_ok=True)

(scripts_dir / "simple_ngs_pipeline.py").write_text('import os\nimport sys\nimport subprocess\nfrom pathlib import Path\n\ndef run_command(command):\n    try:\n        print(f"$ {command}")\n        subprocess.run(command, check=True, shell=True, executable="/bin/bash")\n    except subprocess.CalledProcessError as e:\n        print(f"Error: {e}")\n        sys.exit(1)\n\nif len(sys.argv) != 3:\n    print(f"Usage: {sys.argv[0]} <fastq_file> <reference_genome>")\n    sys.exit(1)\n\nfastq_file = sys.argv[1]\nreference_genome = sys.argv[2]\nfastq_basename = Path(fastq_file).stem\n\n# Run FastQC for quality control\nfastqc_out = "fastqc_out"\nos.makedirs(fastqc_out, exist_ok=True)\nrun_command(f"fastqc -o {fastqc_out} {fastq_file}")\n\n# Run BWA for read alignment\nbwa_out = f"{fastq_basename}.sam"\nrun_command(f"bwa index {reference_genome}")\nrun_command(f"bwa mem {reference_genome} {fastq_file} > {bwa_out}")\n\n# Run SAMtools and bcftools for a simple variant call\nvariant_out = f"{fastq_basename}.vcf"\nrun_command(f"samtools view -bS {bwa_out} > aln.bam")\nrun_command(f"samtools sort aln.bam -o sorted_aln.bam")\nrun_command(f"samtools index sorted_aln.bam")\nrun_command(f"freebayes -f {reference_genome} sorted_aln.bam > {variant_out}")\n\nprint("Pipeline completed successfully!")')
(scripts_dir / "simple_ngs_pipeline.sh").write_text('#!/usr/bin/env bash\nset -euo pipefail\n\nif [ $# -ne 2 ]; then\n  echo "Usage: $0 <fastq_file> <reference_genome>"\n  exit 1\nfi\n\nfastq_file="$1"\nreference_genome="$2"\nfastq_basename=$(basename "$fastq_file" .fastq)\n\n# Run FastQC for quality control\nfastqc_out="fastqc_out"\nmkdir -p "$fastqc_out"\nfastqc -o "$fastqc_out" "$fastq_file"\n\n# Run BWA for read alignment\nbwa_out="${fastq_basename}.sam"\nbwa index "$reference_genome"\nbwa mem "$reference_genome" "$fastq_file" > "$bwa_out"\n\n# Run SAMtools and bcftools for a simple variant call\nvariant_out="${fastq_basename}.vcf"\nsamtools view -bS "$bwa_out" > aln.bam\nsamtools sort aln.bam -o sorted_aln.bam\nsamtools index sorted_aln.bam\nfreebayes -f "$reference_genome" sorted_aln.bam > "$variant_out"\n\necho "Pipeline completed successfully!"')
(scripts_dir / "simple_ngs_pipeline.sh").chmod(0o755)

print("Created:")
for p in sorted(scripts_dir.iterdir()):
    print(" -", p)

### Inspect the Python pipeline

Do not just run scripts. First read them. Ask yourself: what are the inputs, what are the outputs, and what assumptions are made?


In [ ]:
print((Path("script_examples") / "simple_ngs_pipeline.py").read_text())

**Question 2.** List three reasons why this script is not yet a good representation of the workflow you ran in Practicals 2-5.


### Optional: run the toy single-read Python pipeline

This cell runs the simple Python script on read 1 from lane 1. It is intentionally not a full paired-end analysis. It may take a few minutes.


In [ ]:
toy_dir = Path("script_examples/toy_single_read_run")
toy_dir.mkdir(exist_ok=True)

# Use relative paths from the toy directory back to the workdir files.
run("python3 ../simple_ngs_pipeline.py ../../lane1/s-7-1.fastq ../../Saccharomyces_cerevisiae.EF4.68.dna.toplevel.fa", cwd=toy_dir)
run("find . -maxdepth 2 -type f | sort", cwd=toy_dir)

**Question 3.** What output files did the toy script create? Which important outputs from the earlier practicals are missing?


## 3. A paired-end Bash pipeline

Next, the notebook writes a more realistic Bash pipeline. This version uses paired-end reads from both lanes, performs initial and post-trimming FastQC, trims adapters and low-quality sequence, aligns each lane with read groups, merges lanes, marks duplicates, calls variants on chromosome I with GATK and FreeBayes, filters the SNPs, and compares the filtered VCFs.

You do not need to type this into a terminal. The notebook writes the script for you.


In [ ]:
bash_pipeline_text = r'''
#!/usr/bin/env bash
set -euo pipefail

THREADS="${THREADS:-2}"
REF="${1:-Saccharomyces_cerevisiae.EF4.68.dna.toplevel.fa}"
ADAPTERS="${2:-primers_adapters.fa}"

mkdir -p logs qc/raw qc/trimmed trimmed bam variants

# Reference preparation
samtools faidx "$REF"
bwa index -a is "$REF"
DICT="${REF%.*}.dict"
if [ ! -f "$DICT" ]; then
  picard CreateSequenceDictionary R="$REF" O="$DICT"
fi

for lane in lane1 lane2; do
  lane_number="${lane#lane}"
  mkdir -p "qc/raw/${lane}" "qc/trimmed/${lane}" "trimmed/${lane}"

  # Initial QC
  fastqc -o "qc/raw/${lane}" "${lane}/s-7-1.fastq" "${lane}/s-7-2.fastq"

  # Adapter trimming
  trimmomatic PE -phred33 -threads "$THREADS" \
    -trimlog "logs/${lane}.adapter_trim.log" \
    "${lane}/s-7-1.fastq" "${lane}/s-7-2.fastq" \
    "trimmed/${lane}/s-7-1.adapter.paired.fastq" "trimmed/${lane}/s-7-1.adapter.unpaired.fastq" \
    "trimmed/${lane}/s-7-2.adapter.paired.fastq" "trimmed/${lane}/s-7-2.adapter.unpaired.fastq" \
    ILLUMINACLIP:"${ADAPTERS}":2:30:10 MINLEN:36

  # Quality trimming
  trimmomatic PE -phred33 -threads "$THREADS" \
    -trimlog "logs/${lane}.quality_trim.log" \
    "trimmed/${lane}/s-7-1.adapter.paired.fastq" "trimmed/${lane}/s-7-2.adapter.paired.fastq" \
    "trimmed/${lane}/s-7-1.trim.paired.fastq" "trimmed/${lane}/s-7-1.trim.unpaired.fastq" \
    "trimmed/${lane}/s-7-2.trim.paired.fastq" "trimmed/${lane}/s-7-2.trim.unpaired.fastq" \
    LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

  # Post-trimming QC
  fastqc -o "qc/trimmed/${lane}" \
    "trimmed/${lane}/s-7-1.trim.paired.fastq" \
    "trimmed/${lane}/s-7-2.trim.paired.fastq"

  # Alignment, conversion, sorting and indexing
  bwa mem -t "$THREADS" \
    -R "@RG\tID:${lane_number}\tLB:library\tPL:Illumina\tPU:${lane}\tSM:yeast" \
    "$REF" \
    "trimmed/${lane}/s-7-1.trim.paired.fastq" \
    "trimmed/${lane}/s-7-2.trim.paired.fastq" | \
    samtools view -@ "$THREADS" -b - | \
    samtools sort -@ "$THREADS" -o "bam/${lane}.sorted.bam" -

  samtools index "bam/${lane}.sorted.bam"
  samtools flagstat "bam/${lane}.sorted.bam" > "bam/${lane}.flagstat.txt"
done

# Merge lanes and mark duplicates
picard MergeSamFiles INPUT=bam/lane1.sorted.bam INPUT=bam/lane2.sorted.bam OUTPUT=bam/library.bam
samtools index bam/library.bam
picard MarkDuplicates INPUT=bam/library.bam OUTPUT=bam/library_final.bam METRICS_FILE=bam/duplicate_metrics.txt
samtools index bam/library_final.bam
samtools flagstat bam/library_final.bam > bam/library_final.flagstat.txt

# Variant calling on chromosome I with two callers
gatk HaplotypeCaller \
  -R "$REF" -I bam/library_final.bam -L I \
  -mbq 20 --minimum-mapping-quality 50 \
  -O variants/gatk_variants_raw_I_bq20_mq50.vcf

gatk SelectVariants \
  -R "$REF" \
  --variant variants/gatk_variants_raw_I_bq20_mq50.vcf \
  -O variants/gatk_variants_raw_I_bq20_mq50_SNP.vcf \
  --select-type SNP

freebayes -q 20 -m 50 -u \
  -f "$REF" bam/library_final.bam -r I \
  > variants/freebayes_variants_raw_I_bq20_mq50.vcf

gatk SelectVariants \
  -R "$REF" \
  --variant variants/freebayes_variants_raw_I_bq20_mq50.vcf \
  -O variants/freebayes_variants_raw_I_bq20_mq50_SNP.vcf \
  --select-type SNP

# Variant filtering and comparison
vcftools \
  --vcf variants/gatk_variants_raw_I_bq20_mq50_SNP.vcf \
  --minDP 3 --minQ 20 --max-missing 1 \
  --out variants/gatk_variants_I_flt2_nomissing \
  --recode --recode-INFO-all \
  > variants/gatk_filter.log 2>&1

vcftools \
  --vcf variants/freebayes_variants_raw_I_bq20_mq50_SNP.vcf \
  --minDP 3 --minQ 20 --max-missing 1 \
  --out variants/freebayes_variants_I_flt2_nomissing \
  --recode --recode-INFO-all \
  > variants/freebayes_filter.log 2>&1

vcftools \
  --vcf variants/gatk_variants_I_flt2_nomissing.recode.vcf \
  --diff variants/freebayes_variants_I_flt2_nomissing.recode.vcf \
  --diff-site \
  --out variants/compare_filtered \
  > variants/compare_filtered.log 2>&1

echo "Paired-end NGS pipeline completed successfully."
'''

bash_pipeline = Path("paired_ngs_pipeline.sh")
bash_pipeline.write_text(bash_pipeline_text + "\n")
bash_pipeline.chmod(0o755)
print(bash_pipeline.read_text())

**Question 4.** Find the command that runs `bwa mem`. What information is encoded in the read group (`-R`) string, and why does lane 1 need a different read group from lane 2?

**Question 5.** The script begins with `set -euo pipefail`. What kinds of mistakes or failed commands will this help catch?


### Run the paired-end Bash pipeline

This is a complete scripted pipeline. It may take several minutes. If it fails, read the last command printed in the error output and identify which step failed.


In [ ]:
# Run this if your demonstrator asks you to test the script-based pipeline.
# It is commented out by default because the Snakemake pipeline below performs the same analysis
# with better tracking of inputs and outputs.

# run("THREADS=2 bash paired_ngs_pipeline.sh")

**Question 6.** The Bash pipeline can run the analysis, but it still has weaknesses. What would happen if it failed halfway through? How would you know which steps need to be rerun?


## 4. Snakemake: from one rule to a workflow

Snakemake is a workflow manager. Instead of writing a list of commands, you describe rules with inputs and outputs. Snakemake then works out which commands must run to create the requested final files.

We will start small, then build the full workflow.


### 4.1 What happens without a Snakefile?

Run Snakemake in an empty directory and capture the error message.


In [ ]:
empty_dir = Path("empty_snakemake_test")
empty_dir.mkdir(exist_ok=True)
result = run("snakemake -n", cwd=empty_dir, check=False)
print("Return code:", result.returncode)

**Question 7.** What message does Snakemake return when there is no `Snakefile`? What does this tell you about how Snakemake decides what to run?


### 4.2 A first Snakefile

This Snakefile has one rule and no explicit output. It behaves a lot like a shell script.


In [ ]:
first_snakefile = r'''
rule fastqc_a_file:
    shell:
        "fastqc lane1/s-7-1.fastq"
'''

Path("Snakefile.first").write_text(first_snakefile + "\n")
print(Path("Snakefile.first").read_text())
run("snakemake -s Snakefile.first --cores 1")

**Question 8.** What did Snakemake run? What problem would occur if you ran this Snakefile repeatedly?


### 4.3 Add input and output files

Now the rule tells Snakemake what file it needs and what files it should create.


In [ ]:
input_output_snakefile = r'''
rule fastqc_a_file:
    input:
        "lane1/s-7-1.fastq"
    output:
        "lane1/s-7-1_fastqc.html",
        "lane1/s-7-1_fastqc.zip"
    shell:
        "fastqc {input}"
'''

Path("Snakefile.input_output").write_text(input_output_snakefile + "\n")
print(Path("Snakefile.input_output").read_text())

# Remove outputs from the previous demonstration so this rule has to run once.
for old_output in [Path("lane1/s-7-1_fastqc.html"), Path("lane1/s-7-1_fastqc.zip")]:
    if old_output.exists():
        old_output.unlink()

print("First run:")
run("snakemake -s Snakefile.input_output --cores 1")
print("Second run:")
run("snakemake -s Snakefile.input_output --cores 1")

**Question 9.** Why does the second run report that nothing needs to be done?


### 4.4 Use wildcards to run FastQC on four FASTQ files

Wildcards let one rule describe several files. This example runs FastQC on all four raw FASTQ files.


In [ ]:
wildcard_snakefile = r'''
rule all:
    input:
        "lane1/s-7-1_fastqc.html",
        "lane1/s-7-2_fastqc.html",
        "lane2/s-7-1_fastqc.html",
        "lane2/s-7-2_fastqc.html"

rule fastqc_a_file:
    input:
        "{prefix}.fastq"
    output:
        "{prefix}_fastqc.html",
        "{prefix}_fastqc.zip"
    shell:
        "fastqc {input}"
'''

Path("Snakefile.fastqc_wildcards").write_text(wildcard_snakefile + "\n")
print(Path("Snakefile.fastqc_wildcards").read_text())
run("snakemake -s Snakefile.fastqc_wildcards -n --cores 1")
run("snakemake -s Snakefile.fastqc_wildcards --cores 1")

**Question 10.** Which part of the rule is the wildcard? How many jobs did Snakemake need to run for the four FASTQ files?


## 5. Full Snakemake NGS pipeline

The final Snakefile below reproduces the main analysis from Practicals 2-5 as a Snakemake workflow:

1. Run FastQC on raw reads.
2. Trim adapters with Trimmomatic.
3. Trim low-quality bases with Trimmomatic.
4. Run FastQC again on trimmed paired reads.
5. Index the reference genome for samtools, BWA and Picard.
6. Align both lanes with BWA-MEM and read groups.
7. Sort and index BAM files with samtools.
8. Merge lanes and mark duplicates with Picard.
9. Run BAM QC with flagstat and Qualimap.
10. Call chromosome I variants with GATK and FreeBayes.
11. Select SNPs, filter them, and compare callers with VCFtools.


In [ ]:
final_snakefile = r'''
# Snakefile for GMO4 Practical 8
# This workflow starts with the four raw FASTQ files and produces filtered SNP VCFs
# plus a comparison between GATK HaplotypeCaller and FreeBayes on chromosome I.

REF = "Saccharomyces_cerevisiae.EF4.68.dna.toplevel.fa"
ADAPTERS = "primers_adapters.fa"
LANES = ["lane1", "lane2"]
READS = ["1", "2"]
THREADS = 2
REF_DICT = REF.rsplit(".", 1)[0] + ".dict"

rule all:
    input:
        expand("qc/raw/{lane}/s-7-{read}_fastqc.html", lane=LANES, read=READS),
        expand("qc/trimmed/{lane}/s-7-{read}.trim.paired_fastqc.html", lane=LANES, read=READS),
        expand("bam/{lane}.flagstat.txt", lane=LANES),
        "bam/library_final.bam.bai",
        "bam/library_final.flagstat.txt",
        "qc/qualimap_report/genome_results.txt",
        "variants/gatk_variants_I_flt2_nomissing.recode.vcf",
        "variants/freebayes_variants_I_flt2_nomissing.recode.vcf",
        "variants/compare_filtered.diff.sites_in_files"

rule fastqc_raw:
    input:
        "{lane}/s-7-{read}.fastq"
    output:
        html="qc/raw/{lane}/s-7-{read}_fastqc.html",
        zip="qc/raw/{lane}/s-7-{read}_fastqc.zip"
    shell:
        "mkdir -p qc/raw/{wildcards.lane} && fastqc -o qc/raw/{wildcards.lane} {input}"

rule trim_adapters:
    input:
        r1="{lane}/s-7-1.fastq",
        r2="{lane}/s-7-2.fastq",
        adapters=ADAPTERS
    output:
        r1p="trimmed/{lane}/s-7-1.adapter.paired.fastq",
        r1u="trimmed/{lane}/s-7-1.adapter.unpaired.fastq",
        r2p="trimmed/{lane}/s-7-2.adapter.paired.fastq",
        r2u="trimmed/{lane}/s-7-2.adapter.unpaired.fastq"
    log:
        "logs/{lane}.adapter_trim.log"
    threads: THREADS
    shell:
        """
        mkdir -p trimmed/{wildcards.lane} logs
        trimmomatic PE -phred33 -threads {threads} -trimlog {log} \
          {input.r1} {input.r2} \
          {output.r1p} {output.r1u} \
          {output.r2p} {output.r2u} \
          ILLUMINACLIP:{input.adapters}:2:30:10 MINLEN:36
        """

rule trim_quality:
    input:
        r1="trimmed/{lane}/s-7-1.adapter.paired.fastq",
        r2="trimmed/{lane}/s-7-2.adapter.paired.fastq"
    output:
        r1p="trimmed/{lane}/s-7-1.trim.paired.fastq",
        r1u="trimmed/{lane}/s-7-1.trim.unpaired.fastq",
        r2p="trimmed/{lane}/s-7-2.trim.paired.fastq",
        r2u="trimmed/{lane}/s-7-2.trim.unpaired.fastq"
    log:
        "logs/{lane}.quality_trim.log"
    threads: THREADS
    shell:
        """
        trimmomatic PE -phred33 -threads {threads} -trimlog {log} \
          {input.r1} {input.r2} \
          {output.r1p} {output.r1u} \
          {output.r2p} {output.r2u} \
          LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
        """

rule fastqc_trimmed:
    input:
        "trimmed/{lane}/s-7-{read}.trim.paired.fastq"
    output:
        html="qc/trimmed/{lane}/s-7-{read}.trim.paired_fastqc.html",
        zip="qc/trimmed/{lane}/s-7-{read}.trim.paired_fastqc.zip"
    shell:
        "mkdir -p qc/trimmed/{wildcards.lane} && fastqc -o qc/trimmed/{wildcards.lane} {input}"

rule samtools_faidx:
    input:
        REF
    output:
        REF + ".fai"
    shell:
        "samtools faidx {input}"

rule bwa_index:
    input:
        REF
    output:
        REF + ".amb",
        REF + ".ann",
        REF + ".bwt",
        REF + ".pac",
        REF + ".sa"
    shell:
        "bwa index -a is {input}"

rule picard_dict:
    input:
        REF
    output:
        REF_DICT
    shell:
        "picard CreateSequenceDictionary R={input} O={output}"

rule align_lane:
    input:
        ref=REF,
        fai=REF + ".fai",
        bwt=REF + ".bwt",
        dict=REF_DICT,
        r1="trimmed/{lane}/s-7-1.trim.paired.fastq",
        r2="trimmed/{lane}/s-7-2.trim.paired.fastq"
    output:
        bam="bam/{lane}.sorted.bam",
        bai="bam/{lane}.sorted.bam.bai"
    threads: THREADS
    params:
        rg=lambda wildcards: "@RG\\tID:{}\\tLB:library\\tPL:Illumina\\tPU:{}\\tSM:yeast".format(wildcards.lane.replace("lane", ""), wildcards.lane)
    shell:
        """
        mkdir -p bam
        bwa mem -t {threads} -R '{params.rg}' {input.ref} {input.r1} {input.r2} | \
          samtools view -@ {threads} -b - | \
          samtools sort -@ {threads} -o {output.bam} -
        samtools index {output.bam}
        """

rule flagstat_lane:
    input:
        bam="bam/{lane}.sorted.bam",
        bai="bam/{lane}.sorted.bam.bai"
    output:
        "bam/{lane}.flagstat.txt"
    shell:
        "samtools flagstat {input.bam} > {output}"

rule merge_bams:
    input:
        bams=expand("bam/{lane}.sorted.bam", lane=LANES),
        bais=expand("bam/{lane}.sorted.bam.bai", lane=LANES)
    output:
        "bam/library.bam"
    params:
        inputs=lambda wildcards, input: " ".join(f"INPUT={bam}" for bam in input.bams)
    shell:
        "picard MergeSamFiles {params.inputs} OUTPUT={output}"

rule index_library_bam:
    input:
        "bam/library.bam"
    output:
        "bam/library.bam.bai"
    shell:
        "samtools index {input}"

rule mark_duplicates:
    input:
        bam="bam/library.bam",
        bai="bam/library.bam.bai"
    output:
        bam="bam/library_final.bam",
        metrics="bam/duplicate_metrics.txt"
    shell:
        "picard MarkDuplicates INPUT={input.bam} OUTPUT={output.bam} METRICS_FILE={output.metrics}"

rule index_final_bam:
    input:
        "bam/library_final.bam"
    output:
        "bam/library_final.bam.bai"
    shell:
        "samtools index {input}"

rule flagstat_final:
    input:
        bam="bam/library_final.bam",
        bai="bam/library_final.bam.bai"
    output:
        "bam/library_final.flagstat.txt"
    shell:
        "samtools flagstat {input.bam} > {output}"

rule qualimap_bamqc:
    input:
        bam="bam/library_final.bam",
        bai="bam/library_final.bam.bai"
    output:
        "qc/qualimap_report/genome_results.txt"
    params:
        outdir="qc/qualimap_report"
    shell:
        "qualimap bamqc -bam {input.bam} -outdir {params.outdir} --java-mem-size=1G"

rule gatk_haplotypecaller_chrI:
    input:
        ref=REF,
        fai=REF + ".fai",
        dict=REF_DICT,
        bam="bam/library_final.bam",
        bai="bam/library_final.bam.bai"
    output:
        "variants/gatk_variants_raw_I_bq20_mq50.vcf"
    shell:
        """
        mkdir -p variants
        gatk HaplotypeCaller \
          -R {input.ref} -I {input.bam} -L I \
          -mbq 20 --minimum-mapping-quality 50 \
          -O {output}
        """

rule gatk_select_snps:
    input:
        ref=REF,
        fai=REF + ".fai",
        dict=REF_DICT,
        vcf="variants/gatk_variants_raw_I_bq20_mq50.vcf"
    output:
        "variants/gatk_variants_raw_I_bq20_mq50_SNP.vcf"
    shell:
        "gatk SelectVariants -R {input.ref} --variant {input.vcf} -O {output} --select-type SNP"

rule freebayes_chrI:
    input:
        ref=REF,
        fai=REF + ".fai",
        bam="bam/library_final.bam",
        bai="bam/library_final.bam.bai"
    output:
        "variants/freebayes_variants_raw_I_bq20_mq50.vcf"
    shell:
        "freebayes -q 20 -m 50 -u -f {input.ref} {input.bam} -r I > {output}"

rule freebayes_select_snps:
    input:
        ref=REF,
        fai=REF + ".fai",
        dict=REF_DICT,
        vcf="variants/freebayes_variants_raw_I_bq20_mq50.vcf"
    output:
        "variants/freebayes_variants_raw_I_bq20_mq50_SNP.vcf"
    shell:
        "gatk SelectVariants -R {input.ref} --variant {input.vcf} -O {output} --select-type SNP"

rule filter_gatk_snps:
    input:
        "variants/gatk_variants_raw_I_bq20_mq50_SNP.vcf"
    output:
        vcf="variants/gatk_variants_I_flt2_nomissing.recode.vcf",
        log="variants/gatk_filter.log"
    shell:
        """
        vcftools --vcf {input} --minDP 3 --minQ 20 --max-missing 1 \
          --out variants/gatk_variants_I_flt2_nomissing \
          --recode --recode-INFO-all > {output.log} 2>&1
        """

rule filter_freebayes_snps:
    input:
        "variants/freebayes_variants_raw_I_bq20_mq50_SNP.vcf"
    output:
        vcf="variants/freebayes_variants_I_flt2_nomissing.recode.vcf",
        log="variants/freebayes_filter.log"
    shell:
        """
        vcftools --vcf {input} --minDP 3 --minQ 20 --max-missing 1 \
          --out variants/freebayes_variants_I_flt2_nomissing \
          --recode --recode-INFO-all > {output.log} 2>&1
        """

rule compare_filtered_snps:
    input:
        gatk="variants/gatk_variants_I_flt2_nomissing.recode.vcf",
        freebayes="variants/freebayes_variants_I_flt2_nomissing.recode.vcf"
    output:
        sites="variants/compare_filtered.diff.sites_in_files",
        log="variants/compare_filtered.log"
    shell:
        """
        vcftools --vcf {input.gatk} --diff {input.freebayes} --diff-site \
          --out variants/compare_filtered > {output.log} 2>&1
        """
'''

Path("Snakefile").write_text(final_snakefile + "\n")
print(Path("Snakefile").read_text())

### 5.1 Dry run

A dry run asks Snakemake what it would do, without actually running the workflow.


In [ ]:
run("snakemake -n --cores 2")

**Question 11.** How many jobs does Snakemake plan to run? Which rule or rules will run more than once?


### 5.2 Draw the DAG

The DAG shows the dependency structure of the workflow. If `dot` is available, this cell will create `dag.png` and display it.


In [ ]:
result = run("snakemake --dag | dot -Tpng > dag.png", check=False)
if Path("dag.png").exists():
    from IPython.display import Image, display
    display(Image(filename="dag.png"))
else:
    print("No DAG image was produced. Check whether Graphviz/dot is installed.")

**Question 12.** Find the route from a raw FASTQ file to the final comparison file in the DAG. Which intermediate files connect the trimming stage to the alignment stage?


### 5.3 Run the workflow

This runs the full pipeline. It can take several minutes. Use `--cores 2` so Snakemake can run independent jobs in parallel.


In [ ]:
run("snakemake --cores 2 --printshellcmds")

### 5.4 Inspect outputs

Use the next cells to inspect the key outputs of the pipeline.


In [ ]:
run("find qc trimmed bam variants -maxdepth 3 -type f | sort | sed -n '1,120p'")

In [ ]:
print("Final BAM flagstat:")
print(Path("bam/library_final.flagstat.txt").read_text())

print("\nDuplicate metrics, first non-comment lines:")
metrics = Path("bam/duplicate_metrics.txt").read_text().splitlines()
shown = 0
for line in metrics:
    if line and not line.startswith("#"):
        print(line)
        shown += 1
    if shown == 4:
        break

In [ ]:
def count_variants(vcf_path):
    return sum(1 for line in Path(vcf_path).open() if not line.startswith("#"))

vcfs = [
    "variants/gatk_variants_raw_I_bq20_mq50.vcf",
    "variants/gatk_variants_raw_I_bq20_mq50_SNP.vcf",
    "variants/freebayes_variants_raw_I_bq20_mq50.vcf",
    "variants/freebayes_variants_raw_I_bq20_mq50_SNP.vcf",
    "variants/gatk_variants_I_flt2_nomissing.recode.vcf",
    "variants/freebayes_variants_I_flt2_nomissing.recode.vcf",
]

for vcf in vcfs:
    print(f"{vcf}: {count_variants(vcf)} variants")

In [ ]:
compare = Path("variants/compare_filtered.diff.sites_in_files")
print("First lines of filtered comparison file:")
for i, line in enumerate(compare.open()):
    print(line.rstrip())
    if i == 9:
        break

counts = {"B": 0, "1": 0, "2": 0}
with compare.open() as handle:
    header = next(handle)
    for line in handle:
        fields = line.rstrip().split("\t")
        if len(fields) >= 4 and fields[3] in counts:
            counts[fields[3]] += 1

print("\nFiltered SNP comparison counts:")
print("Present in both files:", counts["B"])
print("Only in GATK file:", counts["1"])
print("Only in FreeBayes file:", counts["2"])

**Question 13.** How many filtered SNPs are present in both callers? How many are unique to GATK and how many are unique to FreeBayes?

**Question 14.** Compare your final BAM flagstat output with the outputs you saw in the earlier practicals. Are the mapping results consistent?


## 6. Rerunning and debugging

A key advantage of Snakemake is that it does not rerun completed jobs unless an output is missing or older than its input. The next cells deliberately remove one output and ask Snakemake what it needs to rebuild.


In [ ]:
# Remove the final comparison file only.
target = Path("variants/compare_filtered.diff.sites_in_files")
if target.exists():
    target.unlink()
    print("Removed", target)

run("snakemake -n --cores 2")

**Question 15.** After removing only the final comparison file, which rule or rules does Snakemake plan to rerun? Why does it not rerun FastQC, trimming and alignment?


In [ ]:
# Recreate the removed file.
run("snakemake --cores 2 --printshellcmds")

## 7. Going further

Choose one of the following extensions if you have time:

- Add `samtools depth` as a rule and output mitochondrial coverage to `qc/mito_coverage.txt`.
- Add a rule that extracts `Mito` alignments to `bam/mito.bam` and indexes it.
- Change `THREADS` and rerun a dry run. Which rules can use more than one thread?
- Move `REF`, `LANES`, `READS` and `THREADS` into a separate `config.yaml` file.
- Add a final `multiqc` rule if MultiQC is installed on your VM.

**Question 16.** Which extension did you try, and what rule did you add or modify?
